In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'data' / '1_train_full.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'data' / '1_test.parquet')

In [3]:
# ============================================================
# 1. 수위 차분 파생변수 생성
# ============================================================
# wl_diff_00h = wl_t(현재수위) - wl_t_minus_00h 파생변수 5개 생성

train_full_fe = train_full.copy()
test_fe = test.copy()

print('train 파생변수 추가 전 크기: ', train_full_fe.shape)
print('test 파생변수 추가 전 크기: ', test_fe.shape)

for h in [1, 3, 6, 12, 24]:
    train_full_fe[f"wl_diff_{h}h"] = (
        train_full['wl_t'] - train_full[f"wl_t_minus_{h}h"]
    )
    test_fe[f"wl_diff_{h}h"] = (
        test['wl_t'] - test[f"wl_t_minus_{h}h"]
    )

print('train 파생변수 추가 후 크기: ', train_full_fe.shape)
print('test 파생변수 추가 후 크기: ', test_fe.shape)


train 파생변수 추가 전 크기:  (749657, 25)
test 파생변수 추가 전 크기:  (186263, 24)
train 파생변수 추가 후 크기:  (749657, 30)
test 파생변수 추가 후 크기:  (186263, 29)


In [4]:
# ============================================================
# 2. 강우강도 파생변수 생성
# ============================================================
# imerg_int_0to0h = (imerg_rain_00h - imerg_rain_00h)/hours
# sfc_int_0to0h = (sfc_rain_00h - sfc_rain_00h)/hours

print('train 파생변수 추가 전 크기: ', train_full_fe.shape)
print('test 파생변수 추가 전 크기: ', test_fe.shape)


for df in [train_full_fe, test_fe]:

    df['imerg_int_1to6h'] = (df['imerg_rain_6h'] - df['imerg_rain_1h']) / 5
    df['imerg_int_6to24h'] = (df['imerg_rain_24h'] - df['imerg_rain_6h']) / 18

    df['sfc_int_1to6h'] = (df['sfc_rain_6h'] - df['sfc_rain_1h']) / 5
    df['sfc_int_6to24h'] = (df['sfc_rain_24h'] - df['sfc_rain_6h']) / 18


print('train 파생변수 추가 후 크기: ', train_full_fe.shape)
print('test 파생변수 추가 후 크기: ', test_fe.shape)

train 파생변수 추가 전 크기:  (749657, 30)
test 파생변수 추가 전 크기:  (186263, 29)
train 파생변수 추가 후 크기:  (749657, 34)
test 파생변수 추가 후 크기:  (186263, 33)


In [6]:
# ============================================================
# 3. 데이터셋 저장하기
# ============================================================

train_full_fe.to_parquet(PROJECT_ROOT / 'data' / '2_train_full_fe.parquet')
test_fe.to_parquet(PROJECT_ROOT / 'data' / '2_test_fe.parquet')